<a href="https://colab.research.google.com/github/AbdelrahmanMohamed-Fathy/compilers-assignment-2/blob/main/Compilers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Input and Verification


In [12]:
from pydantic import BaseModel, Field
from typing import Optional
import json

# @title reading regex string
regex = input("enter your regex: ")

In [13]:
# @title function to return the opposite pair of a bracket
def get_pair(bracket: str) -> str:
    match bracket:
        case ")":
            return "("
        case "]":
            return "["
        case _:
            raise ValueError("Received none bracket")

In [14]:
# @title check for correct brackets using a stack
stack = []
for char in regex:
    # adding to stack if opening a bracket
    if char == "(" or char == "[":
        stack.append(char)
    # popping from stack and checking if matching bracket pair
    elif char == ")" or char == "]":
        if stack.pop() != get_pair(char):
            raise ValueError("Invalid regex")

if len(stack) != 0:
    raise ValueError("Invalid regex")

# Generating NFA


In [15]:
# @title Regex model class as a tree node
class RegexTreeNode(BaseModel):
    name: Optional[str] = None
    is_terminating_state: bool = Field(serialization_alias="isTerminatingState")
    children: Optional[dict[str, list["RegexTreeNode"]]] = Field(
        serialization_alias="paths", exclude_if=lambda paths: not paths
    )

In [16]:
# @title NFA model class as a tree structure
class NfaModel(BaseModel):
    root: RegexTreeNode = Field(serialization_alias="startingState")

In [17]:
# @title Parsing functions


def Or_case_handler(
    regex_list: list[str], is_terminating: bool = False
) -> RegexTreeNode:
    initial_node = RegexTreeNode(is_terminating_state=False, children={"Ɛ": []})
    final_node = RegexTreeNode(is_terminating_state=is_terminating, children=None)
    # handle each indivisually and append it to the initial node while passing the final node as the child to all of them
    for regex in regex_list:
        initial_node.children["Ɛ"].append(
            base_case_handler(regex=regex, child=("Ɛ", [final_node]))
        )
    return initial_node


# handling base case of 1 node going to a second node
def base_case_handler(
    regex: str,
    is_terminating: bool = False,
    child: tuple[str, list[RegexTreeNode]] | None = None,
) -> RegexTreeNode:
    return RegexTreeNode(
        is_terminating_state=False,
        children={
            regex: [
                RegexTreeNode(
                    is_terminating_state=is_terminating,
                    children={child[0]: child[1]} if child else None,
                )
            ]
        },
    )

In [18]:
# @title

print(
    Or_case_handler(["a-z", "A-Z", "/"], is_terminating=True).model_dump_json(
        by_alias=True, indent=2
    )
)

{
  "name": null,
  "isTerminatingState": false,
  "paths": {
    "Ɛ": [
      {
        "name": null,
        "isTerminatingState": false,
        "paths": {
          "a-z": [
            {
              "name": null,
              "isTerminatingState": false,
              "paths": {
                "Ɛ": [
                  {
                    "name": null,
                    "isTerminatingState": true
                  }
                ]
              }
            }
          ]
        }
      },
      {
        "name": null,
        "isTerminatingState": false,
        "paths": {
          "A-Z": [
            {
              "name": null,
              "isTerminatingState": false,
              "paths": {
                "Ɛ": [
                  {
                    "name": null,
                    "isTerminatingState": true
                  }
                ]
              }
            }
          ]
        }
      },
      {
        "name": null,
        "isTerminatin

# Optimizing into DFA
